In [1]:
import os
import json
import numpy as np
import soundfile as sf
from superimposition.noise_superimposition import noise_superimposition

In [14]:
# Test noise_superimposition class
_HERE = os.getcwd()  # os.path.dirname(os.path.abspath(__file__))
_REPO_ROOT = os.path.abspath(os.path.join(_HERE, ".."))
_DATA_ROOT = os.path.join(_REPO_ROOT, "data")

_DRIVING_TO_SPEED = {0: 0, 1: 25, 2: 45, 3: 70}
_VENTING_TO_LEVEL = {0: 0, 1: 1, 2: 2, 3: 3}

ns = noise_superimposition(_DATA_ROOT, fs=48000)

speed = 0
window = 0
venting = 4

noise = ns.get_noise(
    speed=speed,
    window=window,
    mics=None,
    use_correction_gains=False
)

ventilation = ns.get_ventilation(
    level=venting,
    window=window,
    mics=None,
    use_correction_gains=False
)

audio_list = [noise, ventilation]
matched_components = ns.match_duration(audio_list=audio_list, fs=ns.fs)

In [15]:
audio_superimposed = np.sum(np.stack(matched_components, axis=0), axis=0)

out_name = f"speed{speed}_window{window}_vent{venting}.wav"
# out_path = os.path.join(synth_dir, out_name)
sf.write(out_name, audio_superimposed, ns.fs)

In [3]:
# Test generate_from_payload function
scenario_id = 1 # scenario["scenario_id"]
driving = 1 # int(req.get("driving", 0))
window = 1 # int(req.get("window", 0))
venting = 2 # int(req.get("venting", 0))

synth_dir = os.path.join(_REPO_ROOT, "synthetic")
meta_dir = os.path.join(_REPO_ROOT, "metadata")
os.makedirs(synth_dir, exist_ok=True)
os.makedirs(meta_dir, exist_ok=True)

ns = noise_superimposition(_DATA_ROOT, fs=48000)

audio_list = []

noise = ns.get_noise(
    speed=driving,
    window=window,
    mics=None,
    use_correction_gains=False,
)
audio_list.append(noise)

if venting != 0:
    ventilation = ns.get_ventilation(
        level=venting,
        window=window,
        mics=None,
        use_correction_gains=False,
    )
    audio_list.append(ventilation)

matched_components = noise_superimposition.match_duration(
    audio_list,
    fs=ns.fs,
)

audio_superimposed = np.sum(np.stack(matched_components, axis=0), axis=0)

out_name = f"{scenario_id}_speed{driving}_window{window}_vent{venting}.wav"
out_path = os.path.join(synth_dir, out_name)

sf.write(out_path, audio_superimposed, ns.fs)

meta = {
    "file": out_name,
    "scenario_id": scenario_id,
    "sample_rate": ns.fs,
    "channels": int(audio_superimposed.shape[1]) if audio_superimposed.ndim == 2 else 1,
    "duration_sec": float(audio_superimposed.shape[0] / ns.fs),
    "driving": driving,
    "window": window,
    "venting": venting,
    "use_correction_gains": False,
    "engine": "hatci-noise-superimposition",
}

meta_name = f"{scenario_id}_speed{driving}_window{window}_vent{venting}.json"
meta_path = os.path.join(meta_dir, meta_name)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

ValueError: Speed condition 1 is not available for noise with window 1.